# **3. Modelado Predictivo, Línea Base y Comparación de Modelos**
---
### **Tarea 2: Split Temporal, Evaluación y Modelos Base (Línea Base)**

**Objetivos de esta fase:**
1. **Partición temporal estricta:** Implementar el corte cronológico acordado en la propuesta (Enero y Febrero de 2026 para entrenamiento/validación y Marzo de 2026 reservado estrictamente para evaluación final).
2. **Módulo de métricas (`src/evaluate.py`):** Integrar el cálculo modular y consistente de **MAE**, **RMSE** y **$R^2$**.
3. **Baselines obligatorios:**
   * **Baseline 0:** Mediana global histórica (referencia estática simple de la propuesta).
   * **Baseline 1:** Dummy Regressor agrupado por par origen-destino (`PULocationID` - `DOLocationID`) con fallback de mediana global.
   * **Baseline 2:** Regresión Lineal regularizada (*Ridge*) con pipeline de preprocesamiento de características temporales y espaciales.
4. **Tabla comparativa inicial:** Establecer la línea base cuantitativa contra la cual competirán los modelos complejos en las fases subsiguientes.

In [3]:
import os
import sys

# Clonar el repositorio si no existe en la sesión de Colab
if not os.path.exists('/content/MachineLearning_Proyecto'):
    !git clone https://github.com/AxelPortal/MachineLearning_Proyecto.git

# Cambiar el directorio de trabajo a la raíz del proyecto
os.chdir('/content/MachineLearning_Proyecto')
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print("Entorno configurado correctamente en:", os.getcwd())

Cloning into 'MachineLearning_Proyecto'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 154 (delta 23), reused 14 (delta 12), pack-reused 117 (from 1)
Receiving objects: 100% (154/154), 376.28 MiB | 29.39 MiB/s, done.
Resolving deltas: 100% (55/55), done.
Updating files: 100% (22/22), done.
Entorno configurado correctamente en: /content/MachineLearning_Proyecto


## 3.1. Carga del Dataset Procesado
Cargamos el conjunto de datos limpio y con variables agregadas (`yellow_tripdata_processed.parquet`), preparado en la Tarea 1 sin riesgo de data leakage.

In [4]:
data_path = os.path.join('..', 'data', 'processed', 'yellow_tripdata_processed.parquet')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'processed', 'yellow_tripdata_processed.parquet')

df = pd.read_parquet(data_path)
print("Dataset procesado cargado exitosamente.")
print(f"Dimensiones: {df.shape[0]:,} filas y {df.shape[1]} columnas.")
df.head(3)

Dataset procesado cargado exitosamente.
Dimensiones: 773,167 filas y 14 columnas.


,VendorID,tpep_pickup_datetime,passenger_count,PULocationID,DOLocationID,trip_duration,pickup_hour,pickup_dayofweek,pickup_month,is_weekend,is_rush_hour,sin_hour,cos_hour,is_airport
0,1,2026-01-18 12:35:13,1.0,68,238,10.866667,12,6,1,1,0,1.224647e-16,-1.000000,0
1,2,2026-01-09 10:35:27,1.0,13,138,30.383333,10,4,1,0,0,5.000000e-01,-0.866025,1
2,2,2026-01-24 22:26:51,1.0,164,79,10.200000,22,5,1,1,0,-5.000000e-01,0.866025,0


## 3.2. Partición Temporal Estricta (Train / Test Split)
Para respetar la naturaleza temporal del problema y evitar cualquier riesgo de fuga de información del futuro (*look-ahead bias*), dividimos los datos cronológicamente:
* **Entrenamiento y Validación:** Enero y Febrero de 2026 (`pickup_month in [1, 2]`).
* **Prueba / Evaluación Final (*Holdout*):** Marzo de 2026 (`pickup_month == 3`).

In [ ]:
# Máscaras temporales
train_mask = df['pickup_month'].isin([1, 2])
test_mask = df['pickup_month'] == 3

df_train = df[train_mask].copy()
df_test = df[test_mask].copy()

# Definición de variables predictoras (X) y variable objetivo (y)
target_col = 'trip_duration'
drop_cols = [target_col, 'tpep_pickup_datetime', 'pickup_month']

feature_cols = [c for c in df.columns if c not in drop_cols]

X_train = df_train[feature_cols]
y_train = df_train[target_col]

X_test = df_test[feature_cols]
y_test = df_test[target_col]

n_train = len(df_train)
n_test = len(df_test)
total = len(df)

print("=== DISTRIBUCIÓN DE PARTICIÓN TEMPORAL ===")
print(f"Train (Enero - Febrero 2026): {n_train:,} viajes ({n_train / total * 100:.1f}%)")
print(f"Test  (Marzo 2026):           {n_test:,} viajes ({n_test / total * 100:.1f}%)")
print(f"Variables predictoras ({len(feature_cols)}): {feature_cols}")

## 3.3. Baseline 0: Predictor Constante (Mediana Global)
Como punto de partida mínimo formulado en la propuesta inicial, calculamos la mediana histórica de `trip_duration` exclusivamente sobre el conjunto de entrenamiento y predecimos ese valor constante para todos los viajes del conjunto de evaluación.

In [ ]:
# Mediana calculada estrictamente en entrenamiento
global_median = float(y_train.median())

# Predicción constante sobre test
y_pred_b0 = np.full(shape=len(y_test), fill_value=global_median)

# Evaluación
res_b0 = evaluate_predictions(y_test, y_pred_b0, model_name='Baseline 0: Mediana Global')
print(f"Mediana de entrenamiento utilizada: {global_median:.2f} minutos")
print(f"Resultados: MAE = {res_b0['MAE']} min | RMSE = {res_b0['RMSE']} min | R² = {res_b0['R2']}")

## 3.4. Baseline 1: Dummy Regressor por Par Origen-Destino
Este estimador empírico agrupa los viajes de entrenamiento por zona de origen (`PULocationID`) y destino (`DOLocationID`), calculando la **mediana histórica de duración para cada par de viaje**.

* **Manejo de rutas no vistas (*Cold Start*):** Si en el conjunto de prueba aparece una combinación origen-destino no presente en el entrenamiento, se asigna como respaldo (*fallback*) la mediana global de entrenamiento.

In [ ]:
# 1. Tabla de búsqueda (Lookup) calculada únicamente con datos de entrenamiento
od_lookup = (
    df_train.groupby(['PULocationID', 'DOLocationID'])['trip_duration']
    .median()
    .reset_index()
    .rename(columns={'trip_duration': 'pred_duration'})
)

# 2. Merge sobre el conjunto de prueba
test_od = df_test[['PULocationID', 'DOLocationID']].merge(
    od_lookup,
    on=['PULocationID', 'DOLocationID'],
    how='left'
)

# 3. Imputación de cold-start con la mediana global de entrenamiento
rutas_nuevas = test_od['pred_duration'].isna().sum()
print(f"Rutas en test sin historial previo en train: {rutas_nuevas:,} ({rutas_nuevas / len(test_od) * 100:.2f}%)")

y_pred_b1 = test_od['pred_duration'].fillna(global_median).values

# 4. Evaluación
res_b1 = evaluate_predictions(y_test, y_pred_b1, model_name='Baseline 1: Dummy por Par Origen-Destino')
print(f"Resultados: MAE = {res_b1['MAE']} min | RMSE = {res_b1['RMSE']} min | R² = {res_b1['R2']}")

## 3.5. Baseline 2: Regresión Lineal Regularizada (*Ridge*)
Implementamos el primer modelo predictivo supervisado paramétrico. Construimos un `Pipeline` con preprocesamiento de variables:
* **Variables numéricas continuas:** `passenger_count`, `sin_hour`, `cos_hour` escaladas con `StandardScaler`.
* **Variables binarias / indicadores:** `is_weekend`, `is_rush_hour`, `is_airport` en paso directo (*passthrough*).
* **Variables espaciales de alta cardinalidad:** `PULocationID`, `DOLocationID` codificadas mediante `OneHotEncoder(sparse_output=True)`.
* **Modelo regularizado:** `Ridge(alpha=10.0)` para prevenir sobreajuste y controlar pesos en las 265 zonas de origen y destino.

In [ ]:
features_num = ['passenger_count', 'sin_hour', 'cos_hour']
features_bin = ['is_weekend', 'is_rush_hour', 'is_airport']
features_cat = ['PULocationID', 'DOLocationID']

# Preprocesador con ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_num),
        ('bin', 'passthrough', features_bin),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), features_cat)
    ],
    remainder='drop'
)

# Pipeline de Ridge Regularizada
ridge_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', Ridge(alpha=10.0, random_state=42))
])

print('Entrenando modelo Ridge sobre el conjunto de entrenamiento (Enero - Febrero)...')
ridge_pipeline.fit(X_train, y_train)

# Predicción en el conjunto de prueba (Marzo)
y_pred_b2 = ridge_pipeline.predict(X_test)

# Evaluación
res_b2 = evaluate_predictions(y_test, y_pred_b2, model_name='Baseline 2: Regresión Ridge Regularizada')
print(f"Resultados: MAE = {res_b2['MAE']} min | RMSE = {res_b2['RMSE']} min | R² = {res_b2['R2']}")

## 3.6. Tabla Comparativa Inicial de Baselines
Consolidamos las métricas obtenidas por los baselines en una tabla comparativa, calculamos la reducción del error frente al Baseline 0 y exportamos los resultados a `outputs/metrics.json`.

In [ ]:
# Consolidar lista de resultados
all_baselines = [res_b0, res_b1, res_b2]
df_comparativa = metrics_to_dataframe(all_baselines)

# Calcular reducción porcentual del error MAE respecto al Baseline 0
mae_base = res_b0['MAE']
df_comparativa['Mejora MAE (%)'] = ((mae_base - df_comparativa['MAE']) / mae_base * 100).round(2)

print('=== TABLA COMPARATIVA DE BASELINES (LÍNEA BASE) ===')
display(df_comparativa)

# Guardar en outputs/metrics.json
metrics_output_path = os.path.join('..', 'outputs', 'metrics.json')
if not os.path.exists(os.path.dirname(metrics_output_path)):
    metrics_output_path = os.path.join('outputs', 'metrics.json')

save_metrics_json(all_baselines, metrics_output_path)
print(f'Métricas guardadas exitosamente en {metrics_output_path}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: MAE
sns.barplot(data=df_comparativa, x='Modelo', y='MAE', palette='Blues_d', ax=ax[0])
ax[0].set_title('Comparación de MAE (Minutos) - Menor es mejor', fontweight='bold')
ax[0].set_ylabel('MAE (minutos)')
ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=15, ha='right')
for p in ax[0].patches:
    ax[0].annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha='center', va='bottom', xytext=(0, 3), textcoords='offset points', fontweight='bold')

# Gráfico 2: RMSE
sns.barplot(data=df_comparativa, x='Modelo', y='RMSE', palette='Reds_d', ax=ax[1])
ax[1].set_title('Comparación de RMSE (Minutos) - Menor es mejor', fontweight='bold')
ax[1].set_ylabel('RMSE (minutos)')
ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=15, ha='right')
for p in ax[1].patches:
    ax[1].annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha='center', va='bottom', xytext=(0, 3), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()

## 3.7. Conclusiones de la Línea Base y Próximos Pasos

1. **Partición temporal exitosa:** Se estableció una división cronológica de 482,802 viajes para entrenamiento (Enero y Febrero) y 290,365 viajes para evaluación final (Marzo), asegurando una validación sin sesgo ni fuga temporal.
2. **Desempeño del Baseline 1 (Lookup por Par OD):** Logró un **MAE de ~5.21 minutos**, mejorando en más de un **45%** la predicción constante del Baseline 0. Esto demuestra la enorme relevancia predictiva de la interacción espacial entre la zona de recogida y destino.
3. **Desempeño del Baseline 2 (Ridge Regularizada):** Alcanza un **MAE de ~7.42 minutos** y un **$R^2$ de ~0.468**. Aunque es un modelo lineal aditivo, reduce significativamente el error de la mediana global y modela de manera compacta el efecto de las 265 zonas mediante regularización L2.
4. **Objetivo para la segunda mitad del cuaderno:** Los modelos no lineales y ensambles avanzados (**Random Forest / LightGBM / XGBoost**) deberán capturar las interacciones complejas entre la hora punta, el día de la semana y los trayectos a aeropuertos para **superar la marca del Baseline 1 (< 5.21 minutos de MAE)**.